In [77]:
import ConnectionConfig as cc
from delta import DeltaTable
from pyspark.sql.functions import col,explode, from_unixtime
cc.setupEnvironment()

In [78]:
spark = cc.startLocalCluster("factRit")
spark.getActiveSession()

In [79]:
cc.set_connectionProfile("VeloDB")

df_operational_rides = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "rides") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

# Load the JSON files from the weather folder
df_weather = spark.read.json("./weather/", multiLine=True)

df_weather = df_weather.withColumn("weather_time", from_unixtime(col("dt")))

# Flatten the DataFrame
df_weather_flat = df_weather.select(
    col("zipCode"),
    col("coord.lon"),
    col("coord.lat"),
    col("main.temp"),
    col("main.pressure"),
    col("main.humidity"),
    col("wind.speed"),
    col("clouds.all"),
    col("weather_time"),
    col("sys.country"),
    col("timezone"),
    col("name"),
    explode(col("weather")).alias("weather")  # Exploding the weather array
)

# Select specific fields from the exploded weather structure
df_weather_flat = df_weather_flat.select(
    col("zipCode"),
    col("lon"),
    col("lat"),
    col("temp"),
    col("pressure"),
    col("humidity"),
    col("speed"),
    col("all").alias("cloudiness"),
    col("weather_time"),  # This is still the original timestamp
    col("country"),
    col("timezone"),
    col("name"),
    col("weather.id").alias("weather_id"),
    col("weather.main").alias("weather_main"),
    col("weather.description").alias("weather_description")
)

# Create a view for SQL queries
df_weather_flat.createOrReplaceTempView("weather_source")

# Show the weather data to confirm it loaded correctly
df_weather_flat.show(truncate=False)

df_operational_stations = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()
    

+-------+------+-------+-----+--------+--------+-----+----------+-------------------+-------+--------+--------------------+----------+------------+-------------------+
|zipCode|lon   |lat    |temp |pressure|humidity|speed|cloudiness|weather_time       |country|timezone|name                |weather_id|weather_main|weather_description|
+-------+------+-------+-----+--------+--------+-----+----------+-------------------+-------+--------+--------------------+----------+------------+-------------------+
|2140   |4.4167|51.2167|15.38|1000    |90      |1.79 |100       |2024-10-07 20:58:17|BE     |7200    |Antwerpen Borgerhout|801       |Clouds      |few clouds         |
|2140   |4.4167|51.2167|15.38|1000    |90      |1.79 |100       |2024-10-07 20:58:17|BE     |7200    |Antwerpen Borgerhout|800       |Clear       |clear sky          |
|2140   |4.4167|51.2167|15.38|1000    |90      |1.79 |100       |2024-10-07 20:58:17|BE     |7200    |Antwerpen Borgerhout|500       |Rain        |light rain   

#### 2 MAKE DIMENSION TABLES AVAILABLE AS VIEWS


In [80]:
dimDate = spark.read.format("delta").load("spark-warehouse/dimDate")
dim_Customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLock")

dim_Customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")
dimDate.createOrReplaceTempView("dimDate")

In [81]:
df_operational_rides.createOrReplaceTempView("rides_source")
df_operational_stations.createOrReplaceTempView("stations_source")

In [82]:
# todo scd doen ## uuid neemt veel ruimte 
fact_rides = spark.sql("""
    SELECT 
        r.rideId AS rideId,
        s1.lockSK AS startLockSK,
        s2.lockSK AS endLockSK,
        r.starttime AS starttime,
        r.endtime AS endtime,
        c.userId AS userId,
        w.zipCode AS zipCode,
        d.dateSK AS dateSK,
        w.weather_main AS weatherCondition,  -- Extracting the main weather condition
        w.weather_description AS weatherDescription  -- Extracting the description
    FROM 
        rides_source AS r
    LEFT OUTER JOIN 
        dimLocks AS s1 ON r.startlockId = s1.lockId
    LEFT OUTER JOIN 
        dimLocks AS s2 ON r.endlockId = s2.lockId
    LEFT OUTER JOIN 
        dimCustomer AS c ON r.subscriptionid = c.userId
    LEFT OUTER JOIN 
        dimDate AS d ON date_format(r.starttime, 'yyyy-MM-dd') = d.calendarDate
    LEFT OUTER JOIN 
        weather_source AS w ON s1.zipCode = w.zipCode 
    WHERE 
        s1.lockId IS NOT NULL AND 
        s2.lockId IS NOT NULL AND 
        c.userId IS NOT NULL
""")

# Show the result
fact_rides.show(100)




+------+--------------------+--------------------+-------------------+-------------------+------+-------+------+----------------+------------------+
|rideId|         startLockSK|           endLockSK|          starttime|            endtime|userId|zipCode|dateSK|weatherCondition|weatherDescription|
+------+--------------------+--------------------+-------------------+-------------------+------+-------+------+----------------+------------------+
|     1|8740727f-4058-4a4...|1b836d32-2b25-465...|2015-09-22 00:00:00|2012-09-22 00:00:00| 13296|   2140|  2455|            Rain|        light rain|
|     1|8740727f-4058-4a4...|1b836d32-2b25-465...|2015-09-22 00:00:00|2012-09-22 00:00:00| 13296|   2140|  2455|           Clear|         clear sky|
|     1|8740727f-4058-4a4...|1b836d32-2b25-465...|2015-09-22 00:00:00|2012-09-22 00:00:00| 13296|   2140|  2455|          Clouds|        few clouds|
|     3|2c7d1a57-c5bd-4f2...|00e7ce4a-4c5e-43a...|2015-09-22 00:00:00|2012-09-22 00:00:00| 25722|   2000| 